In [1]:
import numpy as np
import pandas as pd

In [22]:
# Read only the column names (first row)
data2023 = pd.read_csv(fr"atusact-2023/atusact_2023.dat", dtype={'TRCODE': str})
data2024 = pd.read_csv(fr"atusact-2024/atusact_2024.dat", dtype={'TRCODE': str})
joined_data = pd.concat([data2023, data2024], ignore_index=True)

In [23]:
joined_data['TRCODE']

0         010101
1         181301
2         130133
3         181301
4         020201
           ...  
292650    180209
292651    020201
292652    110101
292653    120303
292654    010101
Name: TRCODE, Length: 292655, dtype: object

In [30]:
cooking = joined_data.where(joined_data['TRCODE'].astype(str) == '010201').dropna()
cooking_duration = cooking['TUACTDUR24']

print("Cooking duration stats:")
print(cooking_duration.describe())
print("\n" + "="*60)
print("Detailed Percentile Analysis:")
print("="*60)

# Calculate various percentiles
percentiles = [1,5,10, 50, 75, 80,  90, 95, 97, 98, 99, 99.5, 99.9, 100]
for p in percentiles:
    val = np.percentile(cooking_duration, p)
    print(f"{p:5.1f}th percentile: {val:7.1f} minutes ({val/60:5.2f} hours)")

print("\n" + "="*60)
print("Outlier Analysis (>95th percentile):")
print("="*60)
threshold_95 = np.percentile(cooking_duration, 95)
outliers = cooking_duration[cooking_duration > threshold_95]
print(f"95th percentile threshold: {threshold_95:.1f} minutes")
print(f"Number of values above 95th percentile: {len(outliers)}")
print(f"Percentage of data: {len(outliers)/len(cooking_duration)*100:.2f}%")
print(f"\nTop 10 longest cooking durations:")
print(cooking_duration.nlargest(10).values)

Cooking duration stats:
count    21010.000000
mean        30.314374
std         26.336437
min          1.000000
25%         15.000000
50%         30.000000
75%         35.000000
max        850.000000
Name: TUACTDUR24, dtype: float64

Detailed Percentile Analysis:
  1.0th percentile:     2.0 minutes ( 0.03 hours)
  5.0th percentile:     5.0 minutes ( 0.08 hours)
 10.0th percentile:    10.0 minutes ( 0.17 hours)
 50.0th percentile:    30.0 minutes ( 0.50 hours)
 75.0th percentile:    35.0 minutes ( 0.58 hours)
 80.0th percentile:    45.0 minutes ( 0.75 hours)
 90.0th percentile:    60.0 minutes ( 1.00 hours)
 95.0th percentile:    60.0 minutes ( 1.00 hours)
 97.0th percentile:    90.0 minutes ( 1.50 hours)
 98.0th percentile:    90.0 minutes ( 1.50 hours)
 99.0th percentile:   120.0 minutes ( 2.00 hours)
 99.5th percentile:   150.0 minutes ( 2.50 hours)
 99.9th percentile:   269.9 minutes ( 4.50 hours)
100.0th percentile:   850.0 minutes (14.17 hours)

Outlier Analysis (>95th percentile)

In [31]:
# Calculate 98th percentile threshold
p98_threshold = np.percentile(cooking_duration, 98)
print(f"98th percentile threshold: {p98_threshold:.1f} minutes ({p98_threshold/60:.2f} hours)")

# Filter data to only include values below 98th percentile
filtered_cooking = cooking_duration[cooking_duration <= p98_threshold]

print(f"\nOriginal data: n={len(cooking_duration)}")
print(f"Filtered data (<=98th percentile): n={len(filtered_cooking)}")
print(f"Removed: {len(cooking_duration) - len(filtered_cooking)} outliers ({(len(cooking_duration) - len(filtered_cooking))/len(cooking_duration)*100:.2f}%)")

print("\n" + "="*60)
print("Statistics comparison:")
print("="*60)
print(f"{'Metric':<20} {'Original':<15} {'Filtered (<98%)':<15}")
print("-"*60)
print(f"{'Mean':<20} {cooking_duration.mean():>10.2f} min   {filtered_cooking.mean():>10.2f} min")
print(f"{'Std Dev':<20} {cooking_duration.std():>10.2f} min   {filtered_cooking.std():>10.2f} min")
print(f"{'Median':<20} {cooking_duration.median():>10.2f} min   {filtered_cooking.median():>10.2f} min")
print(f"{'Min':<20} {cooking_duration.min():>10.2f} min   {filtered_cooking.min():>10.2f} min")
print(f"{'Max':<20} {cooking_duration.max():>10.2f} min   {filtered_cooking.max():>10.2f} min")

print("\n" + "="*60)
print("Recommended distribution parameters (using filtered data):")
print("="*60)
print(f"Mean: {filtered_cooking.mean():.2f} minutes")
print(f"Std:  {filtered_cooking.std():.2f} minutes")


98th percentile threshold: 90.0 minutes (1.50 hours)

Original data: n=21010
Filtered data (<=98th percentile): n=20655
Removed: 355 outliers (1.69%)

Statistics comparison:
Metric               Original        Filtered (<98%)
------------------------------------------------------------
Mean                      30.31 min        28.25 min
Std Dev                   26.34 min        18.93 min
Median                    30.00 min        30.00 min
Min                        1.00 min         1.00 min
Max                      850.00 min        90.00 min

Recommended distribution parameters (using filtered data):
Mean: 28.25 minutes
Std:  18.93 minutes
